# Tokenizer Round-Trip Tests
Verify that encode/decode works correctly on various chord strings.

In [1]:
import sys
sys.path.insert(0, '..')

from src.data.tokenizer import ChordTokenizer, parse_chord

tok = ChordTokenizer()
print(f'Vocab size: {tok.vocab_size}')

Vocab size: 88


## Test parse_chord

In [2]:
test_chords = ['C', 'Amin', 'Fs7', 'Cadd9', 'Bbmin7', 'Gsus4', 'Dno3d', 'E7b9', 'Faugmaj7']

for c in test_chords:
    print(f'  parse_chord({c!r:15s}) -> {parse_chord(c)}')

  parse_chord('C'            ) -> ('C', 'maj', 'none')
  parse_chord('Amin'         ) -> ('A', 'min', 'none')
  parse_chord('Fs7'          ) -> ('Fs', 'dom7', 'none')
  parse_chord('Cadd9'        ) -> ('C', 'maj', 'add9')
  parse_chord('Bbmin7'       ) -> ('As', 'min7', 'none')
  parse_chord('Gsus4'        ) -> ('G', 'sus', 'none')
  parse_chord('Dno3d'        ) -> ('D', 'maj', 'no3d')
  parse_chord('E7b9'         ) -> ('E', 'dom7', '7b9')
  parse_chord('Faugmaj7'     ) -> ('F', 'aug', 'augmaj7')


## Test encode

In [3]:
raw = '<intro_1> C <verse_1> F C E7 Amin Cadd9 Bbmin7'
ids = tok.encode(raw)

print(f'Raw:     {raw}')
print(f'Encoded: {ids}')
print()

# Show each ID and its token name
for i, id_ in enumerate(ids):
    print(f'  ids[{i:2d}] = {id_:3d}  ->  {tok.id2token[id_]}')

Raw:     <intro_1> C <verse_1> F C E7 Amin Cadd9 Bbmin7
Encoded: [1, 4, 39, 51, 66, 5, 44, 51, 66, 39, 51, 66, 43, 53, 66, 48, 52, 66, 39, 51, 68, 49, 55, 66, 2]

  ids[ 0] =   1  ->  [BOS]
  ids[ 1] =   4  ->  <intro>
  ids[ 2] =  39  ->  C
  ids[ 3] =  51  ->  maj
  ids[ 4] =  66  ->  none
  ids[ 5] =   5  ->  <verse>
  ids[ 6] =  44  ->  F
  ids[ 7] =  51  ->  maj
  ids[ 8] =  66  ->  none
  ids[ 9] =  39  ->  C
  ids[10] =  51  ->  maj
  ids[11] =  66  ->  none
  ids[12] =  43  ->  E
  ids[13] =  53  ->  dom7
  ids[14] =  66  ->  none
  ids[15] =  48  ->  A
  ids[16] =  52  ->  min
  ids[17] =  66  ->  none
  ids[18] =  39  ->  C
  ids[19] =  51  ->  maj
  ids[20] =  68  ->  add9
  ids[21] =  49  ->  As
  ids[22] =  55  ->  min7
  ids[23] =  66  ->  none
  ids[24] =   2  ->  [EOS]


## Test decode (round-trip)

In [4]:
decoded = tok.decode(ids)
print(f'Decoded: {decoded}')

Decoded: ['<intro>', 'C', '<verse>', 'F', 'C', 'E7', 'Amin', 'Cadd9', 'Asmin7']


## Test with real dataset sample

In [5]:
from datasets import load_dataset

ds = load_dataset('ailsntua/Chordonomicon', split='train', streaming=True)
sample = next(iter(ds))

print('Raw chords:')
print(sample['chords'])
print()

ids = tok.encode(sample['chords'])
print(f'Encoded length: {len(ids)} tokens')
print(f'Encoded: {ids[:30]}...')
print()

decoded = tok.decode(ids)
print(f'Decoded: {decoded}')

c:\Users\shawn\anaconda3\envs\chord_transformer\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Raw chords:
<intro_1> C <verse_1> F C E7 Amin C F C G7 C F C E7 Amin C F G7 C <verse_2> F C E7 Amin C F C G7 C F C E7 Amin C F G7 C <chorus_1> F C F C G C F C E7 Amin C F G7 C <solo_1> D <chorus_2> G D G D A D G D Fs7 Bmin D G A7 D G A7 D

Encoded length: 209 tokens
Encoded: [1, 4, 39, 51, 66, 5, 44, 51, 66, 39, 51, 66, 43, 53, 66, 48, 52, 66, 39, 51, 66, 44, 51, 66, 39, 51, 66, 46, 53, 66]...

Decoded: ['<intro>', 'C', '<verse>', 'F', 'C', 'E7', 'Amin', 'C', 'F', 'C', 'G7', 'C', 'F', 'C', 'E7', 'Amin', 'C', 'F', 'G7', 'C', '<verse>', 'F', 'C', 'E7', 'Amin', 'C', 'F', 'C', 'G7', 'C', 'F', 'C', 'E7', 'Amin', 'C', 'F', 'G7', 'C', '<chorus>', 'F', 'C', 'F', 'C', 'G', 'C', 'F', 'C', 'E7', 'Amin', 'C', 'F', 'G7', 'C', '<solo>', 'D', '<chorus>', 'G', 'D', 'G', 'D', 'A', 'D', 'G', 'D', 'Fs7', 'Bmin', 'D', 'G', 'A7', 'D', 'G', 'A7', 'D']
